<a href="https://colab.research.google.com/github/shivamkachhadiya/Python_For_Finance_Trading/blob/main/NIFTY_VS_AIRTEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np

holdings = pd.read_csv("holdings.csv")

trades_last = pd.read_csv("Last_1_Year.csv")

trades_prev = pd.read_csv("Prev_Last_Year.csv")

In [24]:
print("HOLDINGS")
print(holdings.shape)
print(holdings.head())

print("\nLAST ONE YEAR TRADES")
print(trades_last.shape)
print(trades_last.head())

print("\nPREVIOUS YEAR TRADES")
print(trades_prev.shape)
print(trades_prev.head())

HOLDINGS
(6, 10)
   Instrument  Qty.  Avg. cost      LTP  Invested  Cur. val     P&L  Net chg.  \
0  BAJFINANCE     3     992.07   913.00   2976.20   2739.00 -237.20     -7.97   
1  BHARTIARTL     3    2104.30  1978.80   6312.90   5936.40 -376.50     -5.96   
2    GOLDBEES    11     120.41   115.64   1324.55   1272.04  -52.51     -3.96   
3   ICICIBANK     3    1428.60  1386.90   4285.80   4160.70 -125.10     -2.92   
4         M&M     2    3224.35  2858.40   6448.70   5716.80 -731.90    -11.35   

   Day chg.  Unnamed: 9  
0     -0.76         NaN  
1      3.20         NaN  
2     -0.42         NaN  
3     -1.15         NaN  
4      0.19         NaN  

LAST ONE YEAR TRADES
(26, 13)
       symbol          isin  trade_date exchange segment series trade_type  \
0  BHARTIARTL  INE397D01024  2025-11-07      NSE      EQ     EQ        buy   
1    GOLDBEES  INF204KB17I5  2025-11-11      NSE      EQ     EQ       sell   
2         M&M  INE101A01026  2025-11-11      NSE      EQ     EQ       sell 

In [25]:
# 1. Remove completely empty columns
holdings = holdings.dropna(axis=1, how="all")

# 2. Combine both trade files
trades = pd.concat(
    [trades_prev, trades_last],
    ignore_index=True
)

# 3. Basic validation function
def validate_data(df, name):
    print(f"\n--- {name} ---")
    print("Rows:", len(df))
    print("Missing values:", df.isna().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

# 4. Validate all datasets
validate_data(holdings, "Holdings")
validate_data(trades, "Combined Trades")

# 5. Check invalid values
print("\nInvalid holding quantities:",
      (holdings["Qty."] < 0).sum())

print("Invalid holding prices:",
      ((holdings["LTP"] < 0) | (holdings["Avg. cost"] < 0)).sum())

print("Invalid trade quantities:",
      (trades["quantity"] <= 0).sum())

print("Invalid trade prices:",
      (trades["price"] <= 0).sum())

print("Unexpected trade types:",
      (~trades["trade_type"].isin(["buy", "sell"])).sum())


--- Holdings ---
Rows: 6
Missing values: 0
Duplicate rows: 0

--- Combined Trades ---
Rows: 61
Missing values: 0
Duplicate rows: 0

Invalid holding quantities: 0
Invalid holding prices: 0
Invalid trade quantities: 0
Invalid trade prices: 0
Unexpected trade types: 0


In [26]:
print(holdings)
total_invested=holdings["Invested"].sum()
total_value=holdings["Cur. val"].sum()
total_pnl=holdings["P&L"].sum()
return_pct=(total_pnl/total_invested)*100

print("Total Invested:", round(total_invested, 2))
print("Current Value:", round(total_value, 2))
print("Total P&L:", round(total_pnl, 2))
print("Return (%):", round(return_pct, 2))

   Instrument  Qty.  Avg. cost      LTP  Invested  Cur. val     P&L  Net chg.  \
0  BAJFINANCE     3     992.07   913.00   2976.20   2739.00 -237.20     -7.97   
1  BHARTIARTL     3    2104.30  1978.80   6312.90   5936.40 -376.50     -5.96   
2    GOLDBEES    11     120.41   115.64   1324.55   1272.04  -52.51     -3.96   
3   ICICIBANK     3    1428.60  1386.90   4285.80   4160.70 -125.10     -2.92   
4         M&M     2    3224.35  2858.40   6448.70   5716.80 -731.90    -11.35   
5  MID150BEES    12     233.93   218.00   2807.10   2616.00 -191.10     -6.81   

   Day chg.  
0     -0.76  
1      3.20  
2     -0.42  
3     -1.15  
4      0.19  
5      0.89  
Total Invested: 24155.25
Current Value: 22440.94
Total P&L: -1714.31
Return (%): -7.1


In [27]:
holdings["Return %"]=(holdings["P&L"]/holdings["Invested"])*100
analysis=(holdings[["Instrument","Invested","P&L","Return %"]])
print(analysis)

   Instrument  Invested     P&L   Return %
0  BAJFINANCE   2976.20 -237.20  -7.969894
1  BHARTIARTL   6312.90 -376.50  -5.963979
2    GOLDBEES   1324.55  -52.51  -3.964365
3   ICICIBANK   4285.80 -125.10  -2.918942
4         M&M   6448.70 -731.90 -11.349574
5  MID150BEES   2807.10 -191.10  -6.807738


In [28]:
import yfinance as yf

In [29]:
nifty=yf.download(
    "^NSEI",
    period="1mo",
    auto_adjust=True
)
print(nifty[["Close"]].tail())

[*********************100%***********************]  1 of 1 completed

Price              Close
Ticker             ^NSEI
Date                    
2026-10-21  22811.349609
2026-10-22  22687.650391
2026-10-23  22600.000000
2026-10-26  22753.199219
2026-10-27  22749.750000


In [30]:
nifty_close=nifty["Close"].squeeze()

nifty_return=nifty_close.pct_change()*100

print(nifty_return.tail())

Date
2026-10-21   -0.339027
2026-10-22   -0.542270
2026-10-23   -0.386335
2026-10-26    0.677873
2026-10-27   -0.015159
Name: ^NSEI, dtype: float64


In [33]:
airtel=yf.download(
    "BHARTIARTL.NS",
    period="1mo",
    auto_adjust=True
)
airtel_close=airtel["Close"].squeeze()
airtel_return=airtel_close.pct_change()*100

print(airtel[["Close"]].tail())
print("\n Airtel Daily Returns:")
print(airtel_return.tail())

[*********************100%***********************]  1 of 1 completed

Price              Close
Ticker     BHARTIARTL.NS
Date                    
2026-10-21   1891.699951
2026-10-22   1895.000000
2026-10-23   1892.099976
2026-10-26   1917.500000
2026-10-27   1978.800049

 Airtel Daily Returns:
Date
2026-10-21    0.879906
2026-10-22    0.174449
2026-10-23   -0.153036
2026-10-26    1.342425
2026-10-27    3.196873
Name: BHARTIARTL.NS, dtype: float64


In [34]:
comparison = pd.DataFrame({
    "Nifty Return %": nifty_return,
    "Airtel Return %": airtel_return
})

print(comparison.tail())

            Nifty Return %  Airtel Return %
Date                                       
2026-10-21       -0.339027         0.879906
2026-10-22       -0.542270         0.174449
2026-10-23       -0.386335        -0.153036
2026-10-26        0.677873         1.342425
2026-10-27       -0.015159         3.196873


In [35]:
covariance=comparison.cov()
print(covariance)

                 Nifty Return %  Airtel Return %
Nifty Return %         0.886695         1.066482
Airtel Return %        1.066482         2.538088


In [36]:
correlation = comparison.corr()

print(correlation)

                 Nifty Return %  Airtel Return %
Nifty Return %         1.000000         0.710907
Airtel Return %        0.710907         1.000000


In [41]:
covariance = comparison.cov()
print("covariance matix is:\n")
print(covariance)


covariance_value = covariance.loc["Airtel Return %", "Nifty Return %"]

print(covariance_value)

nifty_variance = covariance.loc["Nifty Return %", "Nifty Return %"]

beta = covariance_value / nifty_variance

print("Covariance:", covariance_value)
print("Nifty Variance:", nifty_variance)
print("Beta:", beta)

covariance matix is:

                 Nifty Return %  Airtel Return %
Nifty Return %         0.886695         1.066482
Airtel Return %        1.066482         2.538088
1.0664816227132004
Covariance: 1.0664816227132004
Nifty Variance: 0.8866954436233532
Beta: 1.2027597867822315
